In [1]:
#%pip install -r requirements.txt

In [2]:
import time
overall_start = time.perf_counter()

In [3]:
from datetime import date

current_date = date.today()
DATE_RUN = f"{current_date.year}{current_date.month:02d}{current_date.day:02d}"

In [4]:
#some of the libraries do not work with numpy > 2
#pip install -U numpy==1.26.4

In [5]:
#load necessary libraries for the code to run properly
from dataclasses import dataclass, field
from typing import List
from collections import (defaultdict, Counter)
import random
import sys
import pprint
import os
import logging
import string
import math
import uuid
import hashlib

In [6]:
import yaml
import pandas as pd
import rstr
import names
import requests
import simple_icd_10 as icd

In [7]:
### list of random terms for string placeholders ###
base_list = ["lorem ipsum"] 
#["this is a free text data field"]

In [8]:
#read in the model file directly from GitHub
!curl -O https://raw.githubusercontent.com/CBIIT/ctdc-model/refs/heads/prod/model-desc/ctdc_model_file.yaml
#read in the properties file directly from GitHub
!curl -O https://raw.githubusercontent.com/CBIIT/ctdc-model/refs/heads/prod/model-desc/ctdc_model_properties_file.yaml
#read in property_keys
!curl -O https://raw.githubusercontent.com/CBIIT/crdc-ctdc-dataloader/refs/heads/master/config/props_ctdc.yml

#templates need to be downloaded, not able to automate?

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 23896  100 23896    0     0  98798      0 --:--:-- --:--:-- --:--:--  101k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  109k  100  109k    0     0   588k      0 --:--:-- --:--:-- --:--:--  622k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0  2196    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  2196  100  2196    0     0  11649      0 --:

In [9]:
#DEFINE CLASSES
#dataclass for model properties
@dataclass
class ModelProperty:
    name: str = field(default = "")
    desc: str = field(default = "")
    cadsr_code: str = field(default = "")
    value_type: str = field(default = "")
    value_list: List[str] = field(default_factory=list)
    units: List[str] = field(default_factory=list)
    pattern: str = field(default = "")
    url: str = field(default = "")
    req: bool = field(default = False)
    private: bool = field(default = False)
    minimum: int = field(default = 0)
    maximum: int = field(default = 1000)
    exclusiveMinimum:  str = field(default = "")
    exclusiveMaximum:  str = field(default = "")
    weights: List[float]= field(default_factory=list)
    
    def get_weighted_values(self):
        self.weights = []

        if self.name in weight_dict:
            #get property weights for enums if exist in the data_specs_v2 file
            curr_dict = weight_dict[self.name]   
            if True in curr_dict: curr_dict.update({"Yes": curr_dict.pop(True)})
            if False in curr_dict: curr_dict.update({"No": curr_dict.pop(False)})
            
            if (len(self.value_list) - len(curr_dict)) > 0:
                missing_pct: float = round((1-sum(curr_dict.values())) / (len(self.value_list) - len(curr_dict)) , 3)
            else:
                missing_pct = 0.0
            #weight_list = []
            for curr_value in self.value_list:
                if curr_value in curr_dict:
                    self.weights.append(curr_dict[curr_value])
                else:
                    self.weights.append(missing_pct)
        if sum(self.weights) == 0:
            self.weights = []

    def emit_value(self):
        # allows for | separated Race
        if self.name  in ['race'] and "multi_race_opt" in weight_dict:
            uni_choice = random.choices([1,2], weights=weight_dict["multi_race_opt"].values())[0]
        else:
            uni_choice = 1
        
        curr_choice = 1
        
        property_data_value = ""
        output_value = []
        missing_data_list = []

        while curr_choice <= uni_choice:
            self.get_weighted_values()
            
            if self.value_list:
                if len(self.weights) == 0:  #no weights provided
                    property_data_value =  random.choice(self.value_list)
                else:
                    property_data_value = random.choices(self.value_list, weights=self.weights)[0]
                #return property_data_value
            elif "income" in self.name:
                property_data_value = f"${random.uniform(35000, 175000):.2f}"
            elif "first_name" in self.name or "middle_name" in self.name:
                property_data_value = names.get_first_name()
            elif "last_name" in self.name:
                property_data_value = names.get_last_name()
            elif "orcid" in self.name:
                property_data_value = generate_mock_orcid()
            elif "file_location" in self.name:
                characters = string.ascii_letters + string.digits  # Alphanumeric characters
                random_path = ''.join(random.choice(characters) for _ in range(10))
                property_data_value = f"s3://mock_data_{MODEL_VERSION}/data_file/{random_path}"
            elif "_url" in self.name:
                characters = string.ascii_letters + string.digits  # Alphanumeric characters
                random_path = ''.join(random.choice(characters) for _ in range(10))
                if self.name == 'associated_link_url':
                    property_data_value = f"https://www.mock_data_{MODEL_VERSION}/{self.name.replace("_url","")}/mock_clinical_trial/{random_path}"
                else:
                    property_data_value = f"https://www.mock_data_{MODEL_VERSION}/{self.name.replace("_url","")}/{random_path}"
            elif "uuid" in self.name:
                property_data_value = "dg.4DFC/" + str(uuid.uuid4())
            elif "pubmed" in self.name:
                pattern = r'[1-8]{8}'
                property_data_value = rstr.xeger(pattern)
            elif "checksum_value" in self.name:
                data = ''.join(random.choices(string.ascii_letters, k=15))
                data_bytes = data.encode('utf-8')
                md5_hash = hashlib.md5()
                md5_hash.update(data_bytes)
                property_data_value = md5_hash.hexdigest()
            elif self.name == "crdc_id":
                property_data_value = "Value generated during submission"
            elif self.name in custom_data['Numeric_Values'] or self.value_type in ['number', 'integer']:
                #generate a random number between min and max
                if type(self.minimum) == float or type(self.maximum) == float:
                    property_data_value = round(float(random.uniform(float(self.minimum), float(self.maximum))),2)
                elif type(self.minimum) == int or type(self.maximum) == int:
                    property_data_value = round(int(random.uniform(int(self.minimum), int(self.maximum))), 2)
                else:
                    #if no min/max was provided, default to 10-1000 range
                    property_data_value = random.uniform(10.0,1000.0)
            elif self.value_type == 'string':
                property_data_value = random.choice(base_list)
                missing_data_list = missing_data_list + [self.name] 
            elif self.value_type == 'boolean':
                property_data_value = random.choice([True, False])
            
            curr_choice = curr_choice + 1
            output_value = output_value + [property_data_value]
        
        output_value = list(set(output_value)) #get unique records
        if len(output_value) > 1:
            remove_list = ["Not Reported", "Unknown", "Not allowed to collect"]
            for not_allowed in remove_list:
                if not_allowed in output_value:
                    output_value.remove(not_allowed)
                    if len(output_value) == 1:
                        break
            output_value.sort()  #sort values
            output_value = " | ".join(output_value)  #convert to string
        else:
            output_value = output_value[0]
           
        return output_value  #property_data_value

In [10]:
def generate_mock_orcid():
    """Generates a mock ORCID iD with a valid checksum."""

    digits = [random.randint(0, 9) for _ in range(15)]  # Generate 15 random digits

    # Calculate the checksum
    total = 0
    for digit in digits:
        total = (total + digit) * 2
    remainder = total % 11
    checksum = (12 - remainder) % 11

    if checksum == 10:
        checksum_char = 'X'
    else:
        checksum_char = str(checksum)

    orcid_digits = "".join(map(str, digits)) + checksum_char

    # Format the ORCID iD with hyphens
    formatted_orcid = f"{orcid_digits[0:4]}-{orcid_digits[4:8]}-{orcid_digits[8:12]}-{orcid_digits[12:16]}"

    return formatted_orcid

In [11]:
#dataclass for model nodes
@dataclass
class ModelNode:
    name: str = field(default = "")
    properties: List[ModelProperty] = field(default_factory=list)

In [12]:
#dataclass for the ends of a model relationship
@dataclass
class ModelEnds:
    source_node: ModelNode
    destination_node: ModelNode
    multiplicity: str = field(default = "")
    relationship: str = field(default = "")

In [13]:
#dataclass for model relationships
@dataclass
class ModelEdge:
    name: str
    ends_list: List[ModelEnds] = field(default_factory=list)
    properties_list: List[ModelProperty] = field(default_factory=list)

In [14]:
#dataclass for mock data nodes
@dataclass
class DataNode:
    node_id: str
    parent_node_name_list: list
    parent_node_id_list: list
    child_node_id_list: list
    node_type: str
    node_attributes: dict

In [15]:
#dataclass for mock data relationships
@dataclass
class DataEdge:
    edge_id: str
    edge_type: str
    edge_attributes: dict
    source_node: DataNode
    destination_node: DataNode

In [16]:
@dataclass
class Graph:
    dict_of_data_nodes: defaultdict(list) # type: ignore 
    dict_of_data_edges: {} # type: ignore 
    
    def print_data(self, node_type = 'all'):
        if node_type == 'all':
            #for node_type_key in data_graph.dict_of_data_nodes:
            for node_type_key in self.dict_of_data_nodes:
            
                node_values_dict = defaultdict(list)
                df = pd.DataFrame()
                for node in self.dict_of_data_nodes[node_type_key]:
                    node_values_dict['type'].append(node.node_type)
                
                    if  node.parent_node_id_list:
                        node_values_dict['parent_id'].append(node.parent_node_id_list[0])

                    node_values_dict['node_id'].append(node.node_id)
                    for node_prop in node.node_attributes:
                        node_values_dict[node_prop].append(node.node_attributes[node_prop])
                    
                    for node_values_key in node_values_dict:
                        df[node_values_key] = node_values_dict[node_values_key]
                
                    file_name = node_type_key + ".csv"
                    df.to_csv(file_name)
            return
        else:
            if node_type in self.dict_of_data_nodes:
                node_values_dict = defaultdict(list)
                df = pd.DataFrame()
                for node in self.dict_of_data_nodes[node_type]:
                    node_values_dict['type'].append(node.node_type)
                
                    if  node.parent_node_id_list:
                        node_values_dict['parent_id'].append(node.parent_node_id_list[0])
                    
                    node_values_dict['node_id'].append(node.node_id)
                    
                    for node_prop in node.node_attributes:
                        node_values_dict[node_prop].append(node.node_attributes[node_prop])
                    
                    for node_values_key in node_values_dict:
                        df[node_values_key] = node_values_dict[node_values_key]
                    
                    file_name = node_type + ".csv"
                    df.to_csv(file_name)
                return
            else:
                print("node type not found in graph.")
                return
    
    
    def fill_graph(self, model_nodes_dict, model_props_dict,
                   cadsr_df, extra_props):

        missing_list = []
        for node_type in self.dict_of_data_nodes:
            list_of_node_props = model_nodes_dict[node_type].properties
            list_of_data_nodes = self.dict_of_data_nodes[node_type]
            try:
                print(f"Populating node_type for {node_type}")  
               
                for data_node in list_of_data_nodes:
                    for prop in list_of_node_props:
                        remove_list = []
                        #if prop.name in invalid_terms:
                        #    remove_list = invalid_terms[prop.name]["Enum"]
                        
                        check_cadsr = cadsr_df.query(f"property_name == '{prop.name}'")

                        #if prop.name in ['primary_diagnosis_disease_group', 'assessment_timepoint']:
                        #    # the caDSR values fail, default back to the model terms
                        #    pass
                        
                        if prop.name in extra_props['String_Values'] or prop.name in extra_props['Numeric_Values']:
                            if prop.name not in missing_list:
                           #     print(f"No Properties Found, Using Custom User Data file for: {prop.name}")
                                missing_list.append(prop.name)
                            if prop.name in extra_props['String_Values']:
                                if "Enum" in extra_props['String_Values'][prop.name]:
                                    model_props_dict[prop.name].value_list = extra_props['String_Values'][prop.name]["Enum"]
                            else:
                                if "minimum" in extra_props['Numeric_Values'][prop.name]:
                                    model_props_dict[prop.name].minimum = extra_props['Numeric_Values'][prop.name]["minimum"] 
                                    model_props_dict[prop.name].maximum = extra_props['Numeric_Values'][prop.name]["maximum"]   
                        
                        elif len(check_cadsr) > 0:
                            #compare the caDSR terms to the model and get matching
                            model_list = model_props_dict[prop.name].value_list
                            check_list = [i.lower for i in model_list]
                            matching_list = [i.lower() for i in check_cadsr["permissible_value"].to_list() if i in check_list]
                                
                            if len(matching_list) > 0:
                                model_props_dict[prop.name].value_list = matching_list  #terms that match both model and caDSR
                            else:   #no matching terms, use caDSR only
                                model_props_dict[prop.name].value_list = check_cadsr["permissible_value"].to_list()
                           
                        if len(remove_list) > 0:
                            model_props_dict[prop.name].value_list = [i for i in model_props_dict[prop.name].value_list if i not in remove_list]
                    
                        if model_props_dict[prop.name].emit_value() is not None:
                            data_node.node_attributes[prop.name] = model_props_dict[prop.name].emit_value()
                        else:
                            data_node.node_attributes[prop.name] = "term not in model"                        
            except Exception as e:
                print("error found")
                display_error_line(e)
        return
    
    def get_dict_of_data_nodes(self):
        return self.dict_of_data_nodes
    
    def get_dict_of_data_edges(self):
        return self.dict_of_data_edges
    
    def get_in_degree(self, input_node_id):
        for key in self.dict_of_data_nodes:
            for node in dict_of_data_nodes[key]:
                if node.node_id == input_node_id:
                    return len(node.parent_node_id_list)
        return 0
    
    def get_out_degree(self, input_node_id):
        for key in self.dict_of_data_nodes:
            for node in dict_of_data_nodes[key]:
                if node.node_id == input_node_id:
                    return len(node.child_node_id_list)
        return 0

    def summary(self):
        summary = {'Nodes Summary': {}, 'Edges Summary': {}}

        for node_type in self.dict_of_data_nodes:
            node_count = len(dict_of_data_nodes[node_type])
            summary['Nodes Summary'].update({node_type: node_count})
        
        edge_type_list = []
        for edge in dict_of_data_edges.values():
            edge_type_list.append(edge.edge_type)
        edge_type_counter = Counter(edge_type_list)
        for edge_type, edge_type_count in edge_type_counter.items():
            summary['Edges Summary'].update({edge_type: edge_type_count})
        
        return summary
            

In [17]:
def display_error_line(ex):
    trace = []
    tb = ex.__traceback__
    while tb is not None:
        trace.append({"filename": tb.tb_frame.f_code.co_filename, "name": tb.tb_frame.f_code.co_name, "lineno": tb.tb_lineno})
        tb = tb.tb_next
    print(str({'type': type(ex).__name__, 'message': str(ex), 'trace': trace}))

In [18]:
def get_node_level(data):
    relationships = data["Relationships"]
    all_df = pd.DataFrame(columns=["Child" ,"Type", "Parent", "Relation"])
    
    for rel_type in relationships:
        mul = relationships[rel_type]["Mul"]
        ends = relationships[rel_type]["Ends"]
        for curr_row in ends:
            add_list = [curr_row["Src"], rel_type, curr_row["Dst"], mul ]
            all_df.loc[len(all_df)] = add_list
            
    multi_child = list(set(all_df[all_df.duplicated(subset=['Child'], keep='last')]["Child"].tolist()))  
    rel_df = all_df.query(f"Parent not in {multi_child} and Child not in {multi_child}")
            
    all_node_list = list(set(rel_df["Parent"].tolist() + rel_df["Child"].tolist()))
    all_node_df = pd.DataFrame(all_node_list, columns=["Node_Name"])
    all_node_df["Node_Level"] = -1
    
    top_level = rel_df[~rel_df["Parent"].isin(rel_df["Child"])]["Parent"]
    x = all_node_df.merge(top_level, left_on="Node_Name", right_on="Parent", how="left", indicator=True)
    all_node_df.loc[x.query("_merge == 'both'").index, "Node_Level"] = 0
    
    rel_df = add_levels(rel_df, all_node_df, 0)

    return rel_df, all_df, multi_child

In [19]:
def get_col_names(node_data, id_field_data):
    all_col_names = pd.DataFrame(columns=["File Name", "Column Name"])
    key_dict = id_field_data["Properties"]['id_fields']

    all_relations = pd.DataFrame(columns=["Src", "Dst"])
    relationships = node_data["Relationships"]    
    for rel_type in relationships:
        ends = relationships[rel_type]["Ends"]
        all_relations = pd.concat([all_relations, pd.DataFrame(ends)])

    for curr_node in node_data["Nodes"].keys():
        parent_keys = list(all_relations.query(f"Src == '{curr_node}'")["Dst"])
        props = node_data["Nodes"][curr_node]["Props"]
        full_list = ["type"] + [f"{i}.{key_dict[i]}" for i in parent_keys] + props
        if "crdc_id" in full_list:
            full_list.remove("crdc_id")
            
        z = pd.DataFrame({"File Name":curr_node, "Column Name": full_list})        
        all_col_names  = pd.concat([all_col_names, z])
    return all_col_names

In [20]:
def add_levels(rel_df, all_node_df, start_lvl):
    for curr_index in range(start_lvl, start_lvl+5):
       try:
           x = rel_df.merge(all_node_df.query(f"Node_Level == {curr_index}"), left_on="Parent", right_on="Node_Name")  
           x.drop_duplicates("Child", inplace=True)
           if "Node_Level" in x.columns:
               x.drop("Node_Level", axis=1, inplace=True)

           z = all_node_df.merge(x, left_on="Node_Name", right_on="Child", how="left", indicator=True) 
           z = z.query("Node_Level == -1")               
               
           all_node_df.loc[z.query("_merge == 'both'").index, "Node_Level"] = curr_index + 1
       except Exception as e:
           print(e)
           
    rel_df = rel_df.merge(all_node_df, left_on="Child", right_on="Node_Name", how="left")
    rel_df.rename(columns={"Node_Level": "New_Child_Node_Level"}, inplace=True)
    
    rel_df = rel_df.merge(all_node_df, left_on="Parent", right_on="Node_Name", how="left")
    rel_df.rename(columns={"Node_Level": "New_Parent_Node_Level"}, inplace=True)
    
    rel_df = rel_df[["Child", "New_Child_Node_Level", "Type", "Parent", "New_Parent_Node_Level", "Relation"]]
    rel_df = rel_df.sort_values(by = ["New_Parent_Node_Level", "New_Child_Node_Level"], ascending = [True, True])
    
    return rel_df

In [21]:
######BEGIN READ SECTION######
print('BEGIN READ SECTION')
dict_of_model_properties = {}
check_dict = {}

dict_of_model_nodes = {}
model_graph = {}

BEGIN READ SECTION


In [22]:
file_sep = os.path.sep

In [23]:
configuration_files = './ctdc-mock-data-config-files.yaml'

with open(configuration_files) as f:
    configuration_files = yaml.load(f, Loader=yaml.FullLoader)

In [24]:
configuration_files

{'DATA_SPEC_FILE': '.\\ctdc-mock-data-specs_v2.yaml',
 'CUSTOM_TERMS_FILE': '.\\user_custom_terms.yaml',
 'UBERON_LOCATION_FILE': '.\\list_of_UBERON_codes.yaml'}

In [25]:
#READ MODEL FILES AND FILE WITH SYNTHETIC VALUES
NODE_FILE = "ctdc_model_file.yaml"
PROP_FILE = "ctdc_model_properties_file.yaml"
ID_FILE = "props_ctdc.yml"

DATA_SPEC_FILE = configuration_files['DATA_SPEC_FILE']
CUSTOM_TERMS_FILE = configuration_files['CUSTOM_TERMS_FILE']
PRIMARY_SITE = configuration_files['UBERON_LOCATION_FILE']
#INVALID_TERMS = configuration_files['INVALID_CADSR']


In [26]:
#with open(INVALID_TERMS) as f:
#    invalid_terms = yaml.load(f, Loader=yaml.FullLoader)

In [27]:
with open(DATA_SPEC_FILE) as f:
    data_spec = yaml.load(f, Loader=yaml.FullLoader)

In [28]:
with open(CUSTOM_TERMS_FILE, 'r') as f:
    custom_data = yaml.safe_load(f)

In [29]:
weighted_data = custom_data['WeightProperties']    
weight_dict = {}
for weight_name in weighted_data.keys():
    temp_dict = {}
    [temp_dict.update(i) for i in weighted_data[weight_name]]
    weight_dict.update({weight_name: temp_dict})

In [30]:
from pathlib import Path # for file paths
from importlib.metadata import version # check package version
from bento_mdf import MDFReader
version("bento_mdf")

import logging
logging.basicConfig(filename='mdf.log')

ctdc_model = NODE_FILE
ctdc_props = PROP_FILE

mdf = MDFReader(ctdc_model, ctdc_props, handle="CTDC")
mdf.model

props = mdf.model.props

prop_dict = {}
for curr_prop in props:
    if props[curr_prop].values is not None:
        prop_dict[curr_prop[1]] = props[curr_prop].values


In [31]:
with open(PROP_FILE) as f:
    permissible_value_df = pd.DataFrame()
    property_data = yaml.load(f, Loader=yaml.FullLoader)
    for property_name in property_data['PropDefinitions'].keys():
        property_value_code = ""
        name = property_name
        print(f"getting property values for {name}")
        try:
            property_value_code = property_data['PropDefinitions'][property_name]['Term'][0]['Code']
        except Exception:
            print(f"      {name} does not have a code, please check")
            property_value_code = ""
        try:
            prop_version = property_data['PropDefinitions'][property_name]['Term'][0]['Version']
        except Exception:
            prop_version = '0'
   
#        if name in prop_dict:
#            allowed_values = prop_dict[name]
#            curr_df = pd.DataFrame({"permissible_value":allowed_values, "main_ID": property_value_code, 
#                                     "Sub_Value": "", "Sub_ID":""})
#            curr_df["property_name"] = name
#            permissible_value_df = pd.concat([permissible_value_df, curr_df])
            
        if property_value_code != "":
            URL = f'https://cadsrapi.cancer.gov/rad/NCIAPI/1.0/api/DataElements?publicId={property_value_code}'
            headers = {'Accept': 'application/json'}
            response = requests.get(URL, headers=headers)

            if response.status_code == 200:
                data = response.json()
                if len(data["DataElements"]) == 0:   #nothing was found
                    pass
                else:
                    curr_data = data["DataElements"]
                    curr_data = [i for i in curr_data if i["version"] == prop_version]
            
                    if len(curr_data) == 0:
                        curr_data = data["DataElements"][-1]
                    else:
                        curr_data = curr_data[0]
                        
                    if "PermissibleValues" in  curr_data["ValueDomain"]:
                        values = curr_data["ValueDomain"]["PermissibleValues"]    
                        allowed_values = [i["value"] for i in values]
                        main_public_ID = [i["publicId"] for i in values]
                        sub_public_ID = [i["ValueMeaning"]["publicId"] for i in values]
                        sub_longname = [i["ValueMeaning"]["longName"] for i in values]
                    
                        curr_df = pd.DataFrame({"permissible_value":allowed_values, "main_ID": main_public_ID, 
                                     "Sub_Value": sub_longname, "Sub_ID":sub_public_ID})
                        curr_df["property_name"] = name
                        permissible_value_df = pd.concat([permissible_value_df, curr_df])
                    else:
                        pass  #property is in CaDSR but no premissible values (free text / number)
              #  except Exception as e:
              #      print(e)
              #      print(f"PermissibleValues were not found for {property_name} (CDE: {property_value_code})")

getting property values for crdc_id
getting property values for program_name
getting property values for program_short_name
getting property values for program_short_description
getting property values for program_full_description
getting property values for program_sort_order
getting property values for program_external_url
getting property values for study_name
getting property values for study_short_name
getting property values for study_accession
getting property values for study_id
getting property values for nci_id
getting property values for nct_id
getting property values for study_version
getting property values for study_description
getting property values for study_type
getting property values for dates_of_conduct
getting property values for study_outcome
getting property values for other_study_outcome
getting property values for recist_objective_response_rate
getting property values for study_age_group
getting property values for study_population
getting property values for 

In [32]:
permissible_value_df = permissible_value_df[~permissible_value_df['permissible_value'].str.contains('https://', na=False)]
permissible_value_df = permissible_value_df[~permissible_value_df['permissible_value'].str.contains('http://', na=False)]
permissible_value_df.reset_index(inplace=True, drop=True)
#permissible_value_df.to_csv("permissible_value_df.csv")

In [33]:
with open(PROP_FILE) as f:
    value_df = pd.DataFrame()
    property_data = yaml.load(f, Loader=yaml.FullLoader)
    for property_name in property_data['PropDefinitions'].keys():
        try:
            property_value_type = property_data['PropDefinitions'][property_name]['Type']
        except:
            try:
                property_value_type = property_data['PropDefinitions'][property_name]
            except:
                property_value_type = 'string'
        
        name = property_name
        desc = ""
        req= ""
        value_type = ""
        value_list = []
        #synthetic_value_list = []
        units = []
        private = ""
        pattern = ""
        url = ""
        minimum = 0
        maximum = 1000
        exclusiveMinimum = ""
        exclusiveMaximum = ""
        #dist = 'Uniform'
        #weights = []

#        if property_name in string_property_data:
#            if 'Enum' in string_property_data[property_name]:
#                value_list = string_property_data[property_name]["Enum"]
#            if 'minimum' in string_property_data[property_name]:
#                minimum = string_property_data[property_name]['minimum']
#                value_type = 'number'
#            if 'maximum' in string_property_data[property_name]:
#               maximum = string_property_data[property_name]['maximum']
        
        if type(property_value_type) is str:
            name = property_name
            value_type = property_value_type
            if 'Desc' in property_data['PropDefinitions'][property_name]:
                desc = property_data['PropDefinitions'][property_name]['Desc']
            if 'Req' in property_data['PropDefinitions'][property_name]:
                req = property_data['PropDefinitions'][property_name]['Req']
            if 'Private' in property_data['PropDefinitions'][property_name]:
                private = property_data['PropDefinitions'][property_name]['Private']
            if 'minimum' in property_data['PropDefinitions'][property_name]:
                minimum = property_data['PropDefinitions'][property_name]['minimum']
            if 'maximum' in property_data['PropDefinitions'][property_name]:
                maximum = property_data['PropDefinitions'][property_name]['maximum']

        elif type(property_value_type) is list:
            value_type = "list"
            value_list = property_value_type['Enum']
            #add section on reading the url to create a value list if property_value_type contains a url.
            if 'Desc' in property_data['PropDefinitions'][property_name]:
                desc = property_data['PropDefinitions'][property_name]['Desc']
            if 'Req' in property_data['PropDefinitions'][property_name]:
                req = property_data['PropDefinitions'][property_name]['Req']
            if 'Private' in property_data['PropDefinitions'][property_name]:
                private = property_data['PropDefinitions'][property_name]['Private']
            #if 'dist' in property_data['PropDefinitions'][property_name]:
            #    dist = property_data['PropDefinitions'][property_name]['dist']
            #if 'weights' in property_data['PropDefinitions'][property_name]:
            #    weights = property_data['PropDefinitions'][property_name]['weights']

                
        elif type(property_value_type) is dict:
            if 'Desc' in property_data['PropDefinitions'][property_name]:
                desc = property_data['PropDefinitions'][property_name]['Desc']
            if 'value_type' in property_value_type:
                value_type = property_value_type['value_type']
            if "Enum" in property_value_type:
                value_list = property_value_type["Enum"]
            if 'units' in property_value_type:
                units = property_value_type['units']
            if 'pattern' in property_value_type:
                pattern = property_value_type['pattern']
                value_type = "regex"
            if 'Req' in property_data['PropDefinitions'][property_name]:
                req = property_data['PropDefinitions'][property_name]['Req']
            if 'Private' in property_data['PropDefinitions'][property_name]:
                private = property_data['PropDefinitions'][property_name]['Private']
            if 'minimum' in property_data['PropDefinitions'][property_name]:
                minimum = property_data['PropDefinitions'][property_name]['minimum']
            if 'maximum' in property_data['PropDefinitions'][property_name]:
                maximum = property_data['PropDefinitions'][property_name]['maximum']

        new_list = permissible_value_df.query(f"property_name == '{property_name}'")
        if len(new_list) > 0:
            new_list = list(new_list["permissible_value"])
            new_list = [i for i in new_list if "https://" not in i]  #remove websites as permissible

        dict_of_model_properties[property_name] = ModelProperty(name = name, desc = desc, cadsr_code = property_value_code,
                                                                    value_type = value_type, value_list = value_list,
                                                                    units = units, url = url, req = req, private = private, minimum = minimum, maximum = maximum,
                                                                    exclusiveMinimum = exclusiveMinimum, exclusiveMaximum = exclusiveMaximum)
        if value_type == "string":
            check_dict[property_name] = {"name":name, "value_type": value_type}
        

In [34]:
with open(NODE_FILE) as f:
    node_data = yaml.load(f, Loader=yaml.FullLoader)

In [35]:
with open(ID_FILE) as f:
    id_field_data = yaml.load(f, Loader=yaml.FullLoader)

In [36]:
with open(PRIMARY_SITE) as f:
    primary_site_data = yaml.load(f, Loader=yaml.FullLoader)

In [37]:
MODEL_VERSION = node_data["Version"]

all_col_names = get_col_names(node_data, id_field_data)
node_mapping, all_df, multi_child = get_node_level(node_data)
new_nodes = all_df.merge(node_mapping, how="outer")
new_nodes.fillna(-1, inplace=True)

#sub assay nodes have no parents but are parents of assay
assay_nodes = new_nodes.query("Type == 'of_assay'")
new_nodes = new_nodes.query("Type != 'of_assay'")

In [38]:
while min(new_nodes["New_Child_Node_Level"]) == -1:
    for parent_node in list(set(new_nodes["Parent"])):
        parent_df = pd.crosstab(new_nodes.query(f"Parent == '{parent_node}'")["Parent"], 
                                new_nodes.query(f"Parent == '{parent_node}'")["New_Parent_Node_Level"])        
        if parent_df.shape[1] > 1:
            z = new_nodes.query(f"Parent == '{parent_node}'") 
            new_nodes.loc[z.index, "New_Parent_Node_Level"] = int(parent_df.columns[-1])
        
        check_child = new_nodes.query(f"Child == '{parent_node}' and New_Child_Node_Level > 0")
        if len(check_child) > 0 and -1 in parent_df.columns:
            node_level = int(check_child["New_Child_Node_Level"].iloc[0])
            z = new_nodes.query(f"Parent == '{parent_node}'") 
            new_nodes.loc[z.index, "New_Parent_Node_Level"] = node_level
    
        if -1 not in parent_df:
            try:
                child_nodes =  new_nodes.query(f"Parent == '{parent_node}'")
                if len(child_nodes) > 0:
                    uni_child = list(set(child_nodes["Child"]))
                    for curr_child in uni_child:
                        z = new_nodes.query(f"Child in '{curr_child}' and New_Child_Node_Level == -1")
                        if len(z) > 0:
                            if min(z["New_Parent_Node_Level"] > 0):
                                child_level = int(max(z["New_Parent_Node_Level"]))+ 1
                                new_nodes.loc[z.index, "New_Child_Node_Level"] = child_level
            except Exception as e:
                print(e)

assay_level = new_nodes.query("Child == 'assay'")
assay_nodes["New_Child_Node_Level"] = assay_level["New_Child_Node_Level"].iloc[0]
assay_nodes["New_Parent_Node_Level"] = assay_level["New_Child_Node_Level"].iloc[0]-1
new_nodes = pd.concat([new_nodes, assay_nodes])

In [39]:
node_mapping = new_nodes

In [40]:
edge_specs = data_spec['RelationshipSpecs']

node_count = pd.DataFrame(columns = ["Parent", "Child", "Node Count", "Distribution"])
for key in edge_specs.keys():
    for relation in edge_specs[key]:
        try:
            if "NodeCount" in edge_specs[key][relation]:
                new_list = [key, relation, edge_specs[key][relation]['NodeCount'], edge_specs[key][relation]['SrcNodeCount']]  
            else:
                new_list = [key, relation, 0, edge_specs[key][relation]['SrcNodeCount']]  
            node_count.loc[len(node_count)] = new_list
        except Exception as e:
            print(e)
            print(key)
            print(relation)

In [41]:
node_mapping = node_mapping.merge(node_count, how="outer")

In [42]:
node_mapping.sort_values(by=['New_Parent_Node_Level', 
                             'New_Child_Node_Level'], ascending=[True, True], inplace=True)

In [43]:
nodes = node_data['Nodes']

for node_name in nodes.keys():
    #print(node_name, nodes[node_name]['Props'])
    if nodes[node_name]['Props']:
        try:
            property_list = [dict_of_model_properties[property_name] for property_name in nodes[node_name]['Props']]
        except Exception:
            print(f"property: {property_name} has issues for node: {node_name}")
            property_list = []
    else:
        property_list = []
    dict_of_model_nodes[node_name] = ModelNode(name = node_name, properties = property_list)

In [44]:
edges = node_data['Relationships']
curr_index = 1

for edge_name in edges.keys():
    Ends_list = []
    Property_list = []
    edge_multiplicity = edges[edge_name]['Mul']
    print("") 
    for edge_pair in edges[edge_name]['Ends']:
        source_node = edge_pair['Src']
        destination_node = edge_pair['Dst']

      #  if destination_node == "assay" or source_node == "assay":
      #      #need to fix assay node
      #      continue

        curr_map = node_mapping.query(f"Parent == '{destination_node}' and Child == '{source_node}'")
        print(f"{curr_index}: {source_node} {edge_name} {destination_node} : {curr_map['Relation'].item()}")
        curr_index += 1
   
        if 'Mul' in edge_pair:
            edge_multiplicity = edge_pair['Mul']
        try:
            Ends_list.append(ModelEnds(source_node = dict_of_model_nodes[source_node], 
                                       destination_node = dict_of_model_nodes[destination_node], 
                                       multiplicity = edge_multiplicity,
                                       relationship = curr_map['Relation'].item()))
        except Exception as e:
            print(f" model_node: {dict_of_model_nodes[source_node]}")

            print (f"{e} was not found in the model")
    
    if 'Props' in edges[edge_name] and edges[edge_name]['Props']:
        property_list = [dict_of_model_properties[property_name] for property_name in edges[edge_name]['Props']]
    else:
        property_list = []
    
    model_graph[edge_name] = ModelEdge(name = edge_name, ends_list = Ends_list, properties_list = Property_list)
#END READ MODEL FILES
print('END READ SECTION')
######END READ SECTION######


1: participant belongs_to study : many_to_one
2: study belongs_to program : many_to_one
3: participant belongs_to cohort : many_to_one
4: cohort belongs_to study_arm : many_to_one
5: study_arm belongs_to study : many_to_one
6: cohort belongs_to study : many_to_one

7: file associated_with specimen : many_to_one
8: file associated_with diagnosis : many_to_one
9: file associated_with participant : many_to_one
10: file associated_with study : many_to_one
11: associated_link associated_with study : many_to_one
12: image_collection associated_with study : many_to_one
13: file associated_with assay : many_to_one
14: assay associated_with specimen : many_to_one

15: publication of_study study : many_to_many

16: consent_group of_consent_group study : many_to_one

17: assay of_assay next_generation_sequencing_assay : many_to_one
18: assay of_assay immune_assay : many_to_one
19: assay of_assay cell_based_assay : many_to_one

20: demographic of_participant participant : many_to_one
21: exposure

In [45]:
######BEGIN SPAWN SECTION######
print('BEGIN SPAWN SECTION')
dict_of_data_nodes = defaultdict(list)
dict_of_data_edges = {}

BEGIN SPAWN SECTION


In [46]:
#READ DATA SPECS FILE
#FOR BENTO
DATA_SPEC_FILE = configuration_files['DATA_SPEC_FILE']

In [47]:
includeNodes = data_spec['IncludeNodes']

In [48]:
data_spec['HeadNode'][0]["name"]
primary_key = id_field_data['Properties']['id_fields']['program']

In [49]:
#Create head data node object
for head_node in data_spec['HeadNode']:
    head_node_type = head_node['name']
    primary_key = id_field_data['Properties']['id_fields'][head_node_type]
    
    if head_node_type in data_spec['IncludeNodes'].keys():
        logging.error('HeadNode ' + head_node_type + 'is in the IncludeNodes.')
        sys.exit()
    head_node_count = head_node['count']
    id_prefix = head_node['Prefix'] # "HEAD_NODE"
    dst_node_type = head_node_type

    head_node_index = 0
    for count in range(head_node_count):
        if primary_key in custom_data['String_Values']:
            node_id = random.choice(custom_data['String_Values'][primary_key]["Enum"])
        else: 
            node_id =  id_prefix + "-" + str(count+1).zfill(3)
        
        head_node_index += 1
        parent_node_id_list = []
        parent_node_name_list = []
        child_node_id_list = []
        node_type = head_node_type
        node_attributes = {}
        data_node = DataNode(node_id = node_id, parent_node_id_list = parent_node_id_list, 
                             parent_node_name_list = parent_node_name_list, 
                             child_node_id_list = child_node_id_list,
                             node_type = node_type, node_attributes = {})
        dict_of_data_nodes[head_node_type].append(data_node)

In [50]:
for dst_node_type in edge_specs.keys():
    dst_data_nodes_list = dict_of_data_nodes[dst_node_type]

In [51]:
def find_edge_type(node_data, src_node_type, dst_node_type):
    for edge_type in node_data['Relationships']:
        for ends in node_data['Relationships'][edge_type]['Ends']:
            if ends['Src'] == src_node_type and ends['Dst'] == dst_node_type:
               return edge_type
    return None

In [52]:
#node_mapping.fillna(0, inplace=True)
node_mapping["Level_Diff"] = node_mapping["New_Child_Node_Level"] - node_mapping["New_Parent_Node_Level"]
node_mapping = node_mapping.sort_values(by = ["Level_Diff", "New_Parent_Node_Level", "New_Child_Node_Level"], ascending = [True, True, True])

node_list = node_mapping["Parent"].tolist() + node_mapping["Child"].tolist()
ordered_unique_list = list(dict.fromkeys(node_list))

In [53]:
def get_parent_tuple(parent_tuple):
    try:
        parent_tuple = [(i.parent_node_name_list[0], i.parent_node_id_list[0]) 
                        for i in dict_of_data_nodes[parent_tuple[0]] if i.node_id == parent_tuple[1]] 
    except:
        return []
    return parent_tuple[0]

In [54]:
node_mapping

,Child,Type,Parent,Relation,New_Child_Node_Level,New_Parent_Node_Level,Node Count,Distribution,Level_Diff
26,study,belongs_to,program,many_to_one,1.0,0.0,1,fixed,1.0
4,associated_link,associated_with,study,many_to_one,2.0,1.0,1,fixed,1.0
7,consent_group,of_consent_group,study,many_to_one,2.0,1.0,3,fixed,1.0
16,image_collection,associated_with,study,many_to_one,2.0,1.0,5,random,1.0
22,principal_investigator,directs,study,many_to_many,2.0,1.0,3,random,1.0
23,publication,of_study,study,many_to_many,2.0,1.0,5,random,1.0
27,study_arm,belongs_to,study,many_to_one,2.0,1.0,2,fixed,1.0
6,cohort,belongs_to,study_arm,many_to_one,3.0,2.0,1,fixed,1.0
18,participant,belongs_to,cohort,many_to_one,4.0,3.0,50,fixed,1.0
8,demographic,of_participant,participant,many_to_one,5.0,4.0,1,fixed,1.0


In [55]:
#Function to create a skeleton data graph.
#Create a skeleton data graph.
def spawn_nodes():
    study_name = data_spec["StudyName"]
    created_children = []
    children = []
    additional_parents  = pd.DataFrame(columns=["Parent_Node", "Child_Node"])
    for dst_node_type in ordered_unique_list:
        if dst_node_type in dict_of_data_nodes and dst_node_type not in created_children:
            dst_data_nodes_list = dict_of_data_nodes[dst_node_type]
           
            print(f"working on {dst_node_type}: {len(dst_data_nodes_list)} parent nodes already created")
           
            child_nodes = node_mapping.query(f"Parent == '{dst_node_type}'")
            for src_node_type in list(set(child_nodes["Child"])):                
                edge_type = find_edge_type(node_data, src_node_type, dst_node_type)
                map_case = node_mapping.query(f"Child == '{src_node_type}' and Parent == '{dst_node_type}'")
                if len(map_case) == 0:  #this relation does not exist, nothing to check
                    continue
                
                node_counter = int(map_case["Node Count"].item())
                node_distribution = map_case["Distribution"].item()
                
                id_prefix = includeNodes[src_node_type]['Prefix']
                try:
                    id_len = includeNodes[src_node_type]["id_length"]
                except:
                    id_len = 3  #default to 3 digits if not supplied in file
                # random a set of id without duplicate
                #node_index = 0
                parent_node_length = len(dst_data_nodes_list)
                if parent_node_length == 0:
                    continue

                min_nodes = parent_node_length * node_counter
                if min_nodes > 0:
                    min_nodes = math.ceil(math.log(min_nodes, 10))
                if id_len < min_nodes:
                    id_len = min_nodes
                
                test_df = pd.DataFrame()
                if node_distribution == 'random':
                    #node_counter_list = range(node_counter)
                    parent_node_list = [random.randint(1, node_counter) for _ in range(parent_node_length)]
                    y = [[i]*parent_node_list[i] for i in range(parent_node_length)]
                    parent_node_list = [item for sublist in y for item in sublist]
                    try:
                        node_id_number_list = random.sample(range(1, 10**id_len), len(parent_node_list))
                    except Exception:
                        print(id_len)
                        print(parent_node_list)
                    test_df = pd.DataFrame({'parent':parent_node_list, 'new_node':node_id_number_list})
                elif node_distribution == 'fixed':
                    if src_node_type in dict_of_data_nodes:
                        prev_made = len(dict_of_data_nodes[src_node_type])
                    else:
                        prev_made = 0
                    try:
                        node_id_number_list = range(1+prev_made, parent_node_length*node_counter + 1 + prev_made)
                    except Exception:
                        print(id_len)
                        print(node_counter)
                    parent_node_list = list(range(parent_node_length))*node_counter
                    parent_node_list.sort()
                    test_df = pd.DataFrame({'parent':parent_node_list, 'new_node':node_id_number_list})
                else:
                    print(f"    issue between found: {dst_node_type} and {src_node_type} : {node_distribution}")
                    additional_parents.loc[len(additional_parents)] = [dst_node_type, src_node_type]

                if len(test_df) > 0:
                    print(f"    relationship found: {dst_node_type}, {src_node_type},  making {len(test_df)} nodes")

                test_df.reset_index(inplace=True)
                test_df.index += prev_made
                
                for curr_row in test_df.index:
                    parent_index = test_df.loc[curr_row, 'parent']
                    child_id = test_df.loc[curr_row, 'new_node']
                    node_id = study_name + "-" + id_prefix + "-" + str(child_id).zfill(id_len)

                    child_node_id_list = []

                    try:
                        parent_node_name_list = dict_of_data_nodes[src_node_type][curr_row].parent_node_name_list
                        parent_node_id_list = dict_of_data_nodes[src_node_type][curr_row].parent_node_id_list
                    except Exception:
                        parent_node_name_list = []
                        parent_node_id_list = [] 
         
                    if dst_node_type not in parent_node_name_list:
                        parent_node_name_list.append(dst_node_type)
                        parent_node_id_list.append(dst_data_nodes_list[parent_index].node_id)

                    parent_tuple = (dst_node_type, dst_data_nodes_list[parent_index].node_id)
                    loop_count = 0
                    while len(parent_tuple) > 0 and loop_count < 5: 
                        parent_tuple = get_parent_tuple(parent_tuple)
                        if len(parent_tuple) > 0:
                            if parent_tuple[0] not in parent_node_name_list:
                                parent_node_name_list.append(parent_tuple[0])
                                parent_node_id_list.append(parent_tuple[1])
                        loop_count += 1
                        
                    node_type = src_node_type
                    #node_attributes = {}
                    src_data_node = DataNode(node_id = node_id, parent_node_id_list = parent_node_id_list, 
                                             parent_node_name_list = parent_node_name_list, 
                                             child_node_id_list = child_node_id_list,
                                         node_type = node_type, node_attributes = {}) #source node created.
                    dict_of_data_nodes[src_node_type].append(src_data_node) #source node added to the dict of nodes.

                    dst_data_nodes_list[parent_index].child_node_id_list.append(node_id) #add created source node to the child nodes list for dst node.
                    edge_id = "edge" + "_" + str(random.randint(10**5, 10**6))
                    edge_attributes = {}
                    data_edge = DataEdge(edge_id = edge_id, edge_type = edge_type, source_node = src_data_node, 
                                     destination_node = dst_data_nodes_list[parent_index], edge_attributes = edge_attributes) #edge created.
                    dict_of_data_edges[edge_id] = data_edge #edge added to the dict of edges.
                children.append(src_node_type)
            created_children.append(dst_node_type)
    data_graph = Graph(dict_of_data_nodes = dict_of_data_nodes, dict_of_data_edges = dict_of_data_edges)
    return data_graph, additional_parents

In [56]:
#Create skeleton data graph
data_graph, additional_parents = spawn_nodes()

working on program: 1 parent nodes already created
    relationship found: program, study,  making 1 nodes
working on study: 1 parent nodes already created
    relationship found: study, consent_group,  making 3 nodes
    relationship found: study, associated_link,  making 1 nodes
    relationship found: study, image_collection,  making 4 nodes
    relationship found: study, publication,  making 4 nodes
    issue between found: study and participant : decendant
    relationship found: study, study_arm,  making 2 nodes
    issue between found: study and file : decendant
    issue between found: study and cohort : decendant
    relationship found: study, principal_investigator,  making 1 nodes
working on study_arm: 2 parent nodes already created
    relationship found: study_arm, cohort,  making 2 nodes
working on cohort: 2 parent nodes already created
    relationship found: cohort, participant,  making 100 nodes
working on participant: 100 parent nodes already created
    relationship 

In [57]:
for filt_idx in additional_parents.index:
    parent_list = dict_of_data_nodes[additional_parents.loc[filt_idx, "Child_Node"]][0].parent_node_name_list
    if additional_parents.loc[filt_idx, "Parent_Node"] in parent_list:
        additional_parents = additional_parents.drop(filt_idx, axis=0)

In [58]:
for filt_idx in additional_parents.index:
    child_node = additional_parents.loc[filt_idx, "Child_Node"]
    parent_node = additional_parents.loc[filt_idx, "Parent_Node"]
    parent_list = [i.node_id for i in dict_of_data_nodes[parent_node]]

    for curr_node in range(0, len(dict_of_data_nodes[child_node])):
        random_node = random.choice(parent_list)
        old_list = data_graph.dict_of_data_nodes[child_node][curr_node].parent_node_name_list
        old_ids = data_graph.dict_of_data_nodes[child_node][curr_node].parent_node_id_list
        
        data_graph.dict_of_data_nodes[child_node][curr_node].parent_node_name_list = old_list + [parent_node]
        data_graph.dict_of_data_nodes[child_node][curr_node].parent_node_id_list = old_ids + [random_node]

In [59]:
#Examine skeleton data graph
# data_graph.summary()
summary = data_graph.summary()
pprint.pprint(summary)
print('END SPAWN SECTION')
######END SPAWN SECTION######

{'Edges Summary': {'associated_with': 1935,
                   'belongs_to': 105,
                   'directs': 1,
                   'of_consent_group': 3,
                   'of_participant': 1566,
                   'of_study': 4},
 'Nodes Summary': {'assay': 385,
                   'associated_link': 1,
                   'cell_based_assay': 3,
                   'cohort': 2,
                   'consent_group': 3,
                   'demographic': 100,
                   'diagnosis': 217,
                   'exposure': 144,
                   'file': 1547,
                   'image_collection': 4,
                   'immune_assay': 2,
                   'next_generation_sequencing_assay': 2,
                   'non_targeted_therapy': 158,
                   'participant': 100,
                   'participant_status': 100,
                   'principal_investigator': 1,
                   'program': 1,
                   'publication': 4,
                   'radiotherapy': 203,
    

In [60]:
total_nodes_created: int = sum(summary["Nodes Summary"].values())

In [61]:
relationship_node_dict = {}
node_id_field_dict = {}
node_list = []
for parent_id in data_spec['RelationshipSpecs']:
    for node_id in data_spec['RelationshipSpecs'][parent_id]:
        relationship_node = {'parent_id': [], 'parent_id_field': [], 'node_id': node_id}
        relationship_node['parent_id'].append(parent_id)
        if node_id in id_field_data['Properties']['id_fields']:
            relationship_node['node_id_field'] = id_field_data['Properties']['id_fields'][node_id]
        else:
            relationship_node['node_id_field'] ='node_id'
        if parent_id in id_field_data['Properties']['id_fields']:
            relationship_node['parent_id_field'].append(id_field_data['Properties']['id_fields'][parent_id])
        else:
            relationship_node['parent_id_field'].append('parent_id')
        if node_id not in relationship_node_dict.keys():
            relationship_node_dict[node_id] = relationship_node
        else:
            relationship_node_dict[node_id]['parent_id'].append(relationship_node['parent_id'][0])
            relationship_node_dict[node_id]['parent_id_field'].append(relationship_node['parent_id_field'][0])

for node_type in data_graph.dict_of_data_nodes:
    if node_type in id_field_data['Properties']['id_fields']:
        node_id_field_dict[node_type] = id_field_data['Properties']['id_fields'][node_type]

In [62]:
######BEGIN FILL SECTION######
print('BEGIN FILL SECTION')
include_props_list = []

BEGIN FILL SECTION


In [63]:
def get_random_icd10_code():
# caDSR for icd_10_disease_code ('11479873') is free text
# so generate a code from the ICD-10 library
    """Fetches a random valid ICD-10 code using the icd library."""
    try:
        # Get all valid codes in the database (this might take a moment the first time)
        all_codes = list(icd.get_descendants('I')) # Example: get codes from Chapter I (Infectious diseases)
        
        # Select a random code from the list
        if all_codes:
            random_code = random.choice(all_codes)
            #description = icd.get_description(random_code)
            return random_code #, description
        else:
            return "No codes found in the specified range."
    except Exception as e:
        return f"An error occurred: {e}"

In [64]:
# fills all the nodes with appropriate mock data
data_graph.fill_graph(model_nodes_dict = dict_of_model_nodes,       #nodes
                      model_props_dict = dict_of_model_properties,  #properties
                      cadsr_df = permissible_value_df,              #PV values from caDSR  
                      extra_props = custom_data)               #free text filler for neatness

print('END FILL SECTION')
######END FILL SECTION######

Populating node_type for program
Populating node_type for next_generation_sequencing_assay
Populating node_type for immune_assay
Populating node_type for cell_based_assay
Populating node_type for study
Populating node_type for assay
Populating node_type for specimen
Populating node_type for diagnosis
Populating node_type for consent_group
Populating node_type for study_arm
Populating node_type for cohort
Populating node_type for participant
Populating node_type for associated_link
Populating node_type for image_collection
Populating node_type for publication
Populating node_type for principal_investigator
Populating node_type for targeted_therapy
Populating node_type for participant_status
Populating node_type for demographic
Populating node_type for radiotherapy
Populating node_type for exposure
Populating node_type for non_targeted_therapy
Populating node_type for surgery
Populating node_type for file
END FILL SECTION


In [65]:
#data_graph.dict_of_data_nodes["diagnosis"][0].node_attributes

In [66]:
def cleanup_snowmed_terms(df):
    #makes sure that the snowmed code and snowmed term are in alignment
    with open(PROP_FILE) as f:
        property_data = yaml.load(f, Loader=yaml.FullLoader)
        
    terms = property_data["PropDefinitions"]["snomed_disease_code"]["Enum"]
    values = property_data["PropDefinitions"]["snomed_disease_term"]["Enum"]
    
    terms = [ i for i in terms if i not in ["109989006", "2142002", "254633006", "254900004", "28899001", "35917007", "372244006", "55921005a"]]
    values = [ i for i in values if i not in ["Invasive Ductal Carcinoma", "Ovarian Carcinoma"]]
    
    snowmed_df = pd.DataFrame({"Snowmed_Code": terms, "Snowmed_Translation": values})
    
    if "snomed_disease_code" in df.columns:
        my_dict = snowmed_df["Snowmed_Code"]
        list_of_elements =  list(my_dict.items())
        for curr_idx in df.index:
            random_entry = random.choice(list_of_elements)
            df.loc[curr_idx, 'snomed_disease_code'] = snowmed_df.loc[random_entry[0], 'Snowmed_Code']  #actual term
            df.loc[curr_idx, 'snomed_disease_term'] = snowmed_df.loc[random_entry[0], 'Snowmed_Translation']  #meddra code

    return df

In [67]:
def populate_site_location(final_df):  #populates UBERON Codes
# chooses a random UBERON code and correct translation
# ensures that primary_disease_site and icd_o_primary_site are linked terms
    my_dict = primary_site_data["CancerSiteLocation"]
    for curr_idx in final_df.index:
        random_entry = random.choice(list(my_dict.items()))
        if 'primary_disease_site' in final_df.columns:
            final_df.loc[curr_idx, 'primary_disease_site'] = random_entry[1][0]["UBERON Preferred Term"]  #actual term
        if 'surgical_procedure_anatomical_location' in final_df.columns:
            final_df.loc[curr_idx, 'surgical_procedure_anatomical_location'] = random_entry[0]
        if 'anatomical_collection_site' in final_df.columns:
            final_df.loc[curr_idx, 'anatomical_collection_site'] = random_entry[0]
    return final_df
    

In [68]:
def populate_icd10(final_df):
# chooses a random ICD-10 code from a list of valid codes
    if 'icd_10_disease_code' in final_df.columns:
        for curr_index in final_df.index:
            final_df.loc[curr_index, 'icd_10_disease_code'] =  get_random_icd10_code()
    if 'icd_o_3_tissue_morphology' in final_df.columns:
        for curr_index in final_df.index:
            final_df.loc[curr_index, 'icd_o_3_tissue_morphology'] =  get_random_icd10_code()
    if 'icd_o_primary_site' in final_df.columns:
        for curr_index in final_df.index:
            final_df.loc[curr_idx, 'icd_o_primary_site'] = get_random_icd10_code()
    return final_df

In [69]:
def populate_idc_o(final_df):
# randomly selects an ICD-O code from permissible values and its translated value
# this will overwrite the previously created ids and ensure the code and translation match
    
#list of icd_03_disease_code ('13606049' ) that have no matching PV in icd_03_disease_term ('13606067')
    exclude_list = ['9260/0','9652/3','9663/3','8525/3','9709/3','8507/3','8290/3','9290/0','8932/0','8120/2','8560/3','8870/0','8830/0',
'8822/1','9010/0','9581/3','8210/0','8520/3','9104/1','9161/1','8620/1','8530/3','9110/1','8520/2','8744/3','8982/0','9766/1','8140/2','8453/2',
'8161/0','8082/3','8161/3','8171/3','8220/3','8221/3','8251/0','8261/3','8263/2','8323/0','8371/0','8372/0','8373/0','8374/0','8375/0','8401/3',
'9212/0','8323/1','8460/3','8250/3','8265/3','9912/3','9879/3','9819/3','9968/3','9680/1','9475/3','8250/2','8253/2','8519/2','9445/3','9897/3','8991/3','8901/3',
'8852/3','8380/2','9920/3','8140/3','8671/0','9123/0','8251/3','8500/2','9350/1','8070/0','9910/3','9866/3','9873/3','9874/3','8220/0','9252/0',
'8980/3','8503/2','8522/2','8550/0','8550/1','8572/3','8622/1','8721/3','8780/3','8830/3','9084/3','9300/0','9250/3','9330/0','9330/3',
'9430/3','9931/3','8316/1','8272/3','9709/1','8333/3','8472/1','8150/3','9689/3','9805/3','8921/3','9351/1','8540/3','9478/3','8311/3','8507/2',
'8121/3','8409/3','8319/3','8147/0','9252/3','9200/1','8230/2','8832/0','9986/3','9651/3','9473/3','9562/0','8257/3','8254/3',
'8474/1','8620/3','9737/3','8514/3','8959/0','8571/3','8541/3','8380/1','9813/3','9724/3','9807/3','9898/1','9865/3','9869/3','9911/3','9877/3',
'8124/3','9980/3','9251/3','8720/2','9965/3','9967/3','9872/3','9985/3','9948/3','9840/3','9013/0','8370/0','8512/3','8045/3','9180/3','8743/3',
'8821/1','9679/3','9758/3','9757/3','9871/3','9896/3','9801/3','8370/3','8552/3']

    # populates ICD-O codes and keeps translations
    my_dict = permissible_value_df.query("property_name == 'icd_03_disease_code'")

    if len(my_dict) > 0:
        for curr_idx in final_df.index:
            random_entry = my_dict.sample()   
            while  random_entry["permissible_value"].iloc[0] in exclude_list:
                random_entry = my_dict.sample()   #if term in exclude then try again
            final_df.loc[curr_idx, 'icd_03_disease_code'] = random_entry["permissible_value"].iloc[0]  #ICD-0 Code
            final_df.loc[curr_idx, 'icd_03_disease_term'] = random_entry["Sub_Value"].iloc[0]  #actual term
        
    return final_df

In [70]:
def correct_assay_node(df):
    uni_assay = list(set(df["assay_type"]))
    df.reset_index(inplace=True)
    
    header_cols = [i for i in df.columns if ('.' in i and "_assay" in i)]
    selected_columns = df[header_cols].values.flatten()
    unique_values = pd.unique(selected_columns)
    unique_values.sort()

    id_prefix = includeNodes["assay"]['Prefix']
#    primary_key = id_field_data['Properties']['id_fields']["assay"]

    node_id = []
    for curr_row in range(1, len(unique_values)+1):
        node_id +=  [id_prefix + "-" + str(curr_row).zfill(3)]

#    assay_df = pd.DataFrame({"uni_col":unique_values, "node_id":node_id})

    for curr_assay in uni_assay:
        x = df.query(f"assay_type == '{curr_assay}'")
        keep_col = (curr_assay.replace(" ","_") + "_Assay").lower()

        for curr_col in header_cols:
            if keep_col in curr_col:
                pass   #want to keep this record
            else:
                df.loc[x.index, curr_col] = ""
        
#    for curr_assay in header_cols:
#        assay_merge = df[[curr_assay, "assay_record_id", "assay_type"]].merge(assay_df, left_on=curr_assay,
#                                                                              right_on="uni_col", how="left", 
#                                                                              indicator=True)  
#        assay_merge = assay_merge.query("_merge == 'both'")
#        df.loc[assay_merge.index, primary_key] = assay_merge["node_id"]
    #merge final with df to get assay
    return df

In [71]:
def correct_body_surface_area(df):
    if 'body_surface_area' in df.columns:
        df['body_surface_area'] = [round(math.sqrt(float(i)/3600), 2) for i in df['height'] * df['weight']]
    if 'body_mass_index' in df.columns:
        df['body_mass_index'] = round(df['height'] / df['weight'],2)
    return df

In [72]:
def cleanup_age(df):
    if 'participant_age_at_diagnosis' in df.columns and 'age_at_enrollment' in df.columns:
        cleanup = df.query("age_at_enrollment <= participant_age_at_diagnosis")
        for curr_part in cleanup.index:
            max_value = cleanup.loc[curr_part, "age_at_enrollment"]
            df.loc[curr_part, "participant_age_at_diagnosis"] = random.randint(max(5, max_value-15), max_value-1)        
    return df

In [73]:
def make_final_dataframe(node_type):
# converts the data nodes into a pandas dataframe
# corrects related columns based on dependencies
    primary_key = id_field_data['Properties']['id_fields'][node_type]
    key_col = all_col_names.query(f"`Column Name` == '{primary_key}'")
    try:
        col_names = all_col_names.query(f"`File Name` == '{key_col.iloc[0]['File Name']}'")["Column Name"].tolist()
        if node_type == "file":
            col_names += ["data_file_uuid"]
    except Exception:
        print(f"primary key for {node_type} is {primary_key}")
        print(f"Key col is : {key_col}")

    try:
        test_node = data_graph.dict_of_data_nodes[node_type][0]
    except Exception:
        print(node_type)
        print( data_graph.dict_of_data_nodes[node_type])
    
    final_df = pd.DataFrame()
    for test_node in data_graph.dict_of_data_nodes[node_type]:
        test_node.node_attributes["type"] = test_node.node_type
        new_dict = dict(zip(test_node.parent_node_name_list, test_node.parent_node_id_list))
        df = pd.DataFrame([test_node.node_attributes | new_dict | {primary_key: test_node.node_id}])
        #df = df[col_names]
        final_df = pd.concat([final_df, df])

    final_df.reset_index(inplace=True)

    final_df = cleanup_age(final_df)
    final_df = cleanup_snowmed_terms(final_df)
    final_df = populate_icd10(final_df)
    final_df = populate_idc_o(final_df)
    final_df = populate_site_location(final_df)
    final_df = correct_body_surface_area(final_df)

    if "type" in final_df.columns:
        final_df = final_df.query("type == type")

    for curr_parent in test_node.parent_node_name_list:
        full_parent_name = curr_parent + '.' + id_field_data['Properties']['id_fields'][curr_parent]
        #print(f"changing {curr_parent} into {full_parent_name}")
        final_df = final_df.rename(columns = {curr_parent:full_parent_name})

    if node_type == "assay":
        final_df = correct_assay_node(final_df)

    return final_df

In [74]:
#node_dict = {}
#node_list = ['assay', 'file', 'specimen', 'participant', 'cohort', 'study_arm', 'study', 'program', 'diagnosis']
#for curr_node in node_list:
#    parent_list = data_graph.dict_of_data_nodes[curr_node][0].parent_node_name_list
#    node_dict[curr_node] = parent_list

In [75]:
def get_new_parent_list(curr_file, parent_file_level):
# for the file node, gets the correct parent ids based on file_level
    try:        
        index_level = curr_file.parent_node_id_list[curr_file.parent_node_name_list.index(parent_file_level)]
        matching_record = [i for i in data_graph.dict_of_data_nodes[file_level] if i.node_id ==  index_level]
        curr_file.parent_node_id_list = [""]*len(curr_file.parent_node_name_list)  #blank the list
        curr_file.parent_node_id_list[curr_file.parent_node_name_list.index(file_level)] = matching_record[0].node_id
    except Exception:
  #      print(e)
        pass
    
    return curr_file.parent_node_id_list

In [76]:
# A file can be related to one node
# for example a study file is different from a specimen file
# randomizes the file type and assigns the correct primary keys
file_list = ["study", "participant", "diagnosis", "specimen", "assay"]
for curr_idx, value in enumerate(data_graph.dict_of_data_nodes["file"]):
    file_level = random.choice(file_list)
    new_parent_list = get_new_parent_list(value, file_level)
    data_graph.dict_of_data_nodes["file"][curr_idx].parent_node_id_list = new_parent_list

In [77]:
######PRINT DATA FILES######
print('PRINT DATA FILES')
child_node_id_dict = {}
child_node_id_list = []
missing_df = pd.DataFrame(columns=["Node Name", "Property"])
OUTPUT_FOLDER = ".\\Mock_Data_Output\\"

if not os.path.exists(OUTPUT_FOLDER):
    os.mkdir(OUTPUT_FOLDER)

for node_type in data_graph.dict_of_data_nodes:
    
    primary_key = id_field_data['Properties']['id_fields'][node_type]
    key_col = all_col_names.query(f"`Column Name` == '{primary_key}'")
    try:
        col_names = all_col_names.query(f"`File Name` == '{key_col.iloc[0]['File Name']}'")["Column Name"].tolist()
    except Exception:
        pass
 #       print(f"primary key for {node_type} is {primary_key}")
 #       print(f"Key col is : {key_col}")

    #print(f"working on {node_type} file")
    
    final_df = make_final_dataframe(node_type)
    final_df.reset_index(inplace=True, drop=True)
  
    for curr_col in final_df.columns:
        has_missing = final_df[final_df[curr_col].isin(base_list)]
        if len(has_missing) > 0:
            missing_df.loc[len(missing_df)] = [node_type, curr_col]
     #       print(f"   this property has no PV and assumed free text:     {curr_col}")

    file_name = f"mock-{node_type}_{DATE_RUN}_{MODEL_VERSION}.tsv"
#    file_name = f"mock-{node_type}_{MODEL_VERSION}.tsv"

    Output_Folder = f"{OUTPUT_FOLDER}" 
    Study_Name = f"Study_{data_spec['StudyName']}_Version_{MODEL_VERSION}"
    
    file_path = Output_Folder + file_sep + Study_Name + file_sep + file_name
    if not os.path.exists(Output_Folder):
        os.mkdir(Output_Folder)
    if not os.path.exists(Output_Folder + file_sep + Study_Name):
        os.mkdir(Output_Folder + file_sep + Study_Name)

    if node_type == "file":
        x = final_df.query("data_file_format in ['zip', 'gzip', 'tar', 'gz', 'vcf']")
        final_df["data_file_compression_status"] = "Uncompressed"
        final_df.loc[x.index, "data_file_compression_status"] = "Compressed"
        
    if node_type == "principal_investigator":
        final_df["person_record_id"] = final_df["study.study_short_name"] + "-" + final_df["person_last_name"]
    if  "digital_object_id" in final_df.columns:
        pattern = r'https://doi\.org/10\.[1-9]{4}/[a-z]{1}[0-9]{8}'
        for curr_row in final_df.index:
            curr_pattern = rstr.xeger(pattern)
            final_df.loc[curr_row, "digital_object_id"] = curr_pattern
    try:
        print(f"Writing file: {file_name}")
        final_df = final_df[col_names]
        final_df.to_csv(file_path, sep = "\t", index = False, na_rep='NULL')
    except Exception as e:
        print(f"unable to write file: {file_name}")
        print(e)

print('### All Files have been written ###')

PRINT DATA FILES
Writing file: mock-program_20260618_v2.1.0.tsv
Writing file: mock-next_generation_sequencing_assay_20260618_v2.1.0.tsv
Writing file: mock-immune_assay_20260618_v2.1.0.tsv
Writing file: mock-cell_based_assay_20260618_v2.1.0.tsv
Writing file: mock-study_20260618_v2.1.0.tsv
Writing file: mock-assay_20260618_v2.1.0.tsv
Writing file: mock-specimen_20260618_v2.1.0.tsv
Writing file: mock-diagnosis_20260618_v2.1.0.tsv
Writing file: mock-consent_group_20260618_v2.1.0.tsv
Writing file: mock-study_arm_20260618_v2.1.0.tsv
Writing file: mock-cohort_20260618_v2.1.0.tsv
Writing file: mock-participant_20260618_v2.1.0.tsv
Writing file: mock-associated_link_20260618_v2.1.0.tsv
Writing file: mock-image_collection_20260618_v2.1.0.tsv
Writing file: mock-publication_20260618_v2.1.0.tsv
Writing file: mock-principal_investigator_20260618_v2.1.0.tsv
Writing file: mock-targeted_therapy_20260618_v2.1.0.tsv
Writing file: mock-participant_status_20260618_v2.1.0.tsv
Writing file: mock-demographic_2

In [78]:
missing_df.to_csv("missing_terms.csv")

In [79]:
print("cleaning up downloaded files")

cleaning up downloaded files


In [80]:
file_list = ["ctdc_model_file.yaml", "ctdc_model_properties_file.yaml", "props_ctdc.yml"]
for file_path in file_list:
    if os.path.exists(file_path):
        os.remove(file_path)

In [81]:
print("The Mock Data Program took  " +
      f"{time.perf_counter() - overall_start:.4f} seconds")
print(f"There were {total_nodes_created} number of nodes created")

The Mock Data Program took  363.0229 seconds
There were 3627 number of nodes created
